In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
num_epochs = 16
batch_size = 4
learning_rate = 0.001

In [6]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),          # Mirror image
    transforms.RandomCrop(32, padding=4),       # Zoom in/out
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2), # Color variation
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                             download=True, transform=transform_train)

test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                            download=True, transform=transform_test)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

100%|██████████| 170M/170M [00:03<00:00, 43.1MB/s]


In [7]:
classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [8]:
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [9]:
model = ConvNet()
model = model.to(device)

In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

n_total_steps = len(train_loader)
for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (i+1) % 2000 == 0:
            print (f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{n_total_steps}], Loss: {loss.item():.4f}')

print('Finished Training')

Epoch [1/16], Step [2000/12500], Loss: 2.3079
Epoch [1/16], Step [4000/12500], Loss: 2.2890
Epoch [1/16], Step [6000/12500], Loss: 2.3085
Epoch [1/16], Step [8000/12500], Loss: 2.2971
Epoch [1/16], Step [10000/12500], Loss: 2.3009
Epoch [1/16], Step [12000/12500], Loss: 2.2790
Epoch [2/16], Step [2000/12500], Loss: 2.1817
Epoch [2/16], Step [4000/12500], Loss: 1.9218
Epoch [2/16], Step [6000/12500], Loss: 1.9338
Epoch [2/16], Step [8000/12500], Loss: 1.8626
Epoch [2/16], Step [10000/12500], Loss: 1.9688
Epoch [2/16], Step [12000/12500], Loss: 1.6474
Epoch [3/16], Step [2000/12500], Loss: 2.2468
Epoch [3/16], Step [4000/12500], Loss: 1.8643
Epoch [3/16], Step [6000/12500], Loss: 2.1581
Epoch [3/16], Step [8000/12500], Loss: 1.3956
Epoch [3/16], Step [10000/12500], Loss: 2.3698
Epoch [3/16], Step [12000/12500], Loss: 2.0576
Epoch [4/16], Step [2000/12500], Loss: 1.4708
Epoch [4/16], Step [4000/12500], Loss: 2.7602
Epoch [4/16], Step [6000/12500], Loss: 2.1735
Epoch [4/16], Step [8000/125

In [11]:
 with torch.no_grad():
  n_correct = 0
  n_samples = 0
  n_class_correct = [0 for i in range(10)]
  n_class_samples = [0 for i in range(10)]
  for images, labels in test_loader:
    images = images.to(device)
    labels = labels.to(device)
    outputs = model(images)
    _, predicted = torch.max(outputs, 1)
    n_samples += labels.size(0)
    n_correct += (predicted == labels).sum().item()

    for i in range(batch_size):
      label = labels[i]
      pred = predicted[i]
      if (label == pred):
        n_class_correct[label] += 1
      n_class_samples[label] += 1

  acc = 100.0 * n_correct/ n_samples
  print(f'Accuracy of the network: {acc}%')

  for i in range(10):
    acc = 100.0 * n_class_correct[i] / n_class_samples[i]
    print(f'Accuracy of {classes[i]}: {acc}%')

Accuracy of the network: 60.43%
Accuracy of plane: 70.3%
Accuracy of car: 74.4%
Accuracy of bird: 45.2%
Accuracy of cat: 39.5%
Accuracy of deer: 49.2%
Accuracy of dog: 51.7%
Accuracy of frog: 65.7%
Accuracy of horse: 70.1%
Accuracy of ship: 69.0%
Accuracy of truck: 69.2%
